In [13]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import kagglehub

# General setting. Do not change TEST_SIZE
RANDOM_SEED = 42
TEST_SIZE = 0.3

# Load dataset (from kagglehub)
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
data = pd.read_csv(f"{path}/creditcard.csv")
data['Class'] = data['Class'].astype(int)

# Prepare data
data = data.drop(['Time'], axis=1)
data['Amount'] = StandardScaler().fit_transform(data['Amount'].values.reshape(-1, 1))

fraud = data[data['Class'] == 1]
nonfraud = data[data['Class'] == 0]
print(f'Fraudulent:{len(fraud)}, non-fraudulent:{len(nonfraud)}')
print(f'the positive class (frauds) percentage: {len(fraud)}/{len(fraud) + len(nonfraud)} ({len(fraud)/(len(fraud) + len(nonfraud))*100:.3f}%)')

X = np.asarray(data.iloc[:, ~data.columns.isin(['Class'])])
Y = np.asarray(data.iloc[:, data.columns == 'Class'])

# Split training set and test set
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=TEST_SIZE, random_state=RANDOM_SEED)

# Calculate the ratio of non-fraudulent to fraudulent transactions
f_rate = len(nonfraud) / len(fraud)


Fraudulent:492, non-fraudulent:284315
the positive class (frauds) percentage: 492/284807 (0.173%)


In [2]:
%pip install xgboost

   ---------------------------------------- 0.0/150.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/150.0 MB ? eta -:--:--
   ---------------------------------------- 0.8/150.0 MB 2.1 MB/s eta 0:01:11
   ---------------------------------------- 1.0/150.0 MB 2.2 MB/s eta 0:01:09
   ---------------------------------------- 1.8/150.0 MB 2.6 MB/s eta 0:00:58
    --------------------------------------- 2.4/150.0 MB 2.6 MB/s eta 0:00:57
    --------------------------------------- 2.9/150.0 MB 2.5 MB/s eta 0:00:59
    --------------------------------------- 3.1/150.0 MB 2.4 MB/s eta 0:01:01
    --------------------------------------- 3.7/150.0 MB 2.4 MB/s eta 0:01:00
   - -------------------------------------- 4.2/150.0 MB 2.4 MB/s eta 0:01:01
   - -------------------------------------- 4.7/150.0 MB 2.5 MB/s eta 0:01:00
   - -------------------------------------- 5.2/150.0 MB 2.5 MB/s eta 0:00:59
   - -------------------------------------- 5.8/150.0 MB 2.5 MB/s eta 0:00:59


In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    # Modify hyperparameters here
    # Number of trees (weak learners); more trees may improve performance but increase computation
    n_estimators=200,
    # Controls the contribution of each tree; smaller values often improve generalization
    learning_rate=0.15,
    # Maximum depth of each tree to control model complexity and overfitting
    max_depth=10,
    # Fraction of samples used per tree to help prevent overfitting
    subsample=1,
    # Fraction of features used per tree; helps reduce overfitting and feature noise
    colsample_bytree=0.75,
    # Minimum loss reduction required to make a further partition; larger values make the model more conservative
    gamma=0.5,
    # Handling class imbalance: adjust weight ratio of positive and negative classes to improve minority detection
    scale_pos_weight=f_rate,
    # Minimum sum of instance weight needed in a child node; prevents overfitting to small samples
    min_child_weight=1,
    # Use histogram-based training, which speeds up training and is suitable for large datasets
    tree_method='hist',
    # Evaluation metric for training; logloss evaluates prediction probability accuracy
    eval_metric='logloss',
    # Random seed for reproducibility
    random_state=RANDOM_SEED
)

xgb_model.fit(X_train, y_train)

# Define evaluation function
def evaluation(y_true, y_pred, model_name="Model"):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print(f'\n{model_name} Evaluation:')
    print('===' * 15)
    print('         Accuracy:', accuracy)
    print('  Precision Score:', precision)
    print('     Recall Score:', recall)
    print('         F1 Score:', f1)
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

# Predict and print results
y_pred = xgb_model.predict(X_test)
evaluation(y_test, y_pred, model_name="XGBClassifier")



XGBClassifier Evaluation:
         Accuracy: 0.9996605924417448
  Precision Score: 0.928
     Recall Score: 0.8529411764705882
         F1 Score: 0.8888888888888888

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     85307
           1       0.93      0.85      0.89       136

    accuracy                           1.00     85443
   macro avg       0.96      0.93      0.94     85443
weighted avg       1.00      1.00      1.00     85443



In [14]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report
from sklearn.cluster import KMeans
from sklearn.utils import resample
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import kagglehub

# General setting. Do not change TEST_SIZE
RANDOM_SEED = 42
TEST_SIZE = 0.3

# Load dataset (from kagglehub)
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
data = pd.read_csv(f"{path}/creditcard.csv")
data['Class'] = data['Class'].astype(int)

# Prepare data
data.drop('Time', axis=1, inplace=True)

scaler = RobustScaler()
data['Amount'] = scaler.fit_transform(data[['Amount']])

# Extract features and labels
X = data.drop('Class', axis=1).values
y = data['Class'].values

fraud = X[y == 1]
nonfraud = X[y == 0]

# Undersampling: make the number of normal and fraudulent samples equal
nonfraud_down = resample(nonfraud, replace=False, n_samples=len(fraud), random_state=RANDOM_SEED)
X_resampled = np.vstack([nonfraud_down, fraud])
y_resampled = np.hstack([np.zeros(len(fraud)), np.ones(len(fraud))])

# Split the dataset into training and testing sets (with stratification)
X_train_resample, X_test_resample, y_train_resample, y_test_resample = train_test_split(
   X_resampled, y_resampled, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y_resampled
)

# Apply PCA for dimensionality reduction on training data
pca = PCA().fit(X_train_resample)
n_components = np.argmax(np.cumsum(pca.explained_variance_ratio_) >= 0.95) + 1
pca = PCA(n_components=n_components, random_state=RANDOM_SEED)
X_train_pca = pca.fit_transform(X_train_resample)
X_test_pca = pca.transform(X_test_resample)


In [15]:
# Set KMeans parameters
kmeans_p = {
    'n_clusters': 9,          # Number of clusters
    'init': 'k-means++',      # Initialization method for better convergence
    'n_init': 45,             # Number of times the algorithm will be run with different centroid seeds
}
n_clusters = kmeans_p['n_clusters']

# Build KMeans model (unsupervised learning)
kmeans = KMeans(**kmeans_p, random_state=RANDOM_SEED)

# Train KMeans using only normal samples (labeled as 0)
# Normal samples represent the majority and can serve as a baseline
kmeans.fit(X_train_pca[y_train_resample == 0])

# Classify test data and get assigned cluster labels
cluster_labels = kmeans.predict(X_test_pca)

# Define a function: evaluate the fraud rate within each cluster
def rating(y_pred, n_clusters):
    rate = np.zeros(len(y_pred), dtype=float)  # Create result array of the same length as y_pred
    for i in range(n_clusters):  # Evaluate each cluster index
        mask = (y_pred == i)     # Find test samples assigned to cluster i
        if np.sum(mask) > 0:
            # Compute the proportion of actual fraud cases in this cluster (i.e., fraud probability)
            rate[mask] = np.mean(y_test_resample[mask])
        else:
            rate[mask] = 0       # If no samples in this cluster, consider it non-fraudulent
    return rate

# Evaluate all clusters and calculate fraud rate for each
fraud_rate = rating(cluster_labels, n_clusters)

# Compute the mean fraud rate across all clusters
mean_rate = np.mean(fraud_rate)

# Identify anomaly clusters: those with above-average fraud rates
anomaly_clusters = [
    i for i, rate in enumerate(fraud_rate) if rate > mean_rate
]

# Create predicted labels: if a test sample belongs to an anomaly cluster → predict as fraud (1), otherwise normal (0)
y_pred = np.array([1 if label in anomaly_clusters else 0 for label in cluster_labels])

# Define evaluation function to print common metrics
def evaluation(y_true, y_pred, model_name="Model"):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print(f'\n{model_name} Evaluation:')
    print('===' * 15)
    print('         Accuracy:', accuracy)
    print('  Precision Score:', precision)
    print('     Recall Score:', recall)
    print('         F1 Score:', f1)
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

# Evaluate the performance of the KMeans model on the test set
evaluation(y_test_resample, y_pred, model_name="KMeans (Unsupervised)")



KMeans (Unsupervised) Evaluation:
         Accuracy: 0.543918918918919
  Precision Score: 0.5448275862068965
     Recall Score: 0.5337837837837838
         F1 Score: 0.5392491467576792

Classification Report:
              precision    recall  f1-score   support

         0.0       0.54      0.55      0.55       148
         1.0       0.54      0.53      0.54       148

    accuracy                           0.54       296
   macro avg       0.54      0.54      0.54       296
weighted avg       0.54      0.54      0.54       296

